<a href="https://colab.research.google.com/github/avocado-planet/04-Human-in-the-loop/blob/main/04_hitl_middleware.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LangChain HumanInTheLoopMiddleware 実践

エージェントが危険なツールを呼ぼうとしたとき、**人間の承認を挟んでから実行する**仕組みを学びます。

## このノートブックで学ぶこと

1. HITL（Human-in-the-Loop）の基本構造
2. `checkpointer` と `thread_id` の役割
3. 3つの判断パターン: **approve**（承認）/ **reject**（拒否）/ **edit**（編集）
4. 安全なツールと危険なツールの区別

---
## 1. セットアップ

In [1]:
!pip install --pre -U langchain langgraph langchain-openai -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.7/112.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.8/169.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 2.9 MB/s eta 0:00:00


In [2]:
import os
from getpass import getpass
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

MODEL_NAME = "openai:gpt-4.1-mini"

print(f"モデル: {MODEL_NAME}")

モデル: openai:gpt-4.1-mini


---
## 2. ツール定義

2種類のツールを用意します:
- `get_weather` — 安全な操作（読み取りのみ）→ 承認不要
- `delete_data` — 危険な操作（データ削除）→ 承認必要

In [3]:
from langchain_core.tools import tool


@tool
def get_weather(city: str) -> str:
    """指定した都市の天気を取得します（安全な操作）。"""
    return f"{city}の天気: 晴れ, 25度C"


@tool
def delete_data(target: str) -> str:
    """データを削除します（危険な操作）。"""
    return f"'{target}' を削除しました"


print("ツール定義完了")
print(f"  - {get_weather.name}: 承認不要")
print(f"  - {delete_data.name}: 承認必要")

ツール定義完了
  - get_weather: 承認不要
  - delete_data: 承認必要


---
## 3. エージェント作成

### 重要な構成要素

```
create_agent(
    model=...,
    tools=[...],
    checkpointer=InMemorySaver(),  ← (A) 状態を保存する仕組み
    middleware=[
        HumanInTheLoopMiddleware(  ← (B) どのツールで中断するか
            interrupt_on={...}
        )
    ],
)
```

#### (A) checkpointer とは?

エージェントが中断（interrupt）したとき、**その時点の State（メッセージ履歴、
ツール呼び出し情報など）をまるごと保存**する仕組みです。

人間が判断を返すまでには時間がかかります。その間、エージェントの実行は
止まっているので、状態がどこかに保存されていないと再開できません。

- `InMemorySaver()` — メモリ上に保存（デモ・開発用）
- 本番では DB ベースの checkpointer を使う

#### (B) interrupt_on とは?

ツールごとに「中断するかどうか」を設定します:
- `True` — 全判断（approve/edit/reject）を許可して中断
- `False` — 中断しない（自動承認）
- `{"allowed_decisions": [...]}` — 許可する判断を細かく指定

In [11]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


agent = create_agent(
    model=MODEL_NAME,
    tools=[get_weather, delete_data],
    system_prompt="ユーザーが削除を要求したら、必ず delete_data ツールを使って実行してください。",

    # --- checkpointer ---
    # interrupt 時に State を保存し、後から再開できるようにする
    checkpointer=InMemorySaver(),

    # --- middleware ---
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                # delete_data → 中断して人間の判断を待つ
                "delete_data": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                # get_weather → 中断しない（自動実行）
                "get_weather": False,
            }
        ),
    ],
)

print("エージェント作成完了")

エージェント作成完了


---
## 4. thread_id とは?

checkpointer が State を保存するとき、**どのセッションの State か**を識別するための ID です。

```python
config = {"configurable": {"thread_id": "session-1"}}
```

- 同じ `thread_id` → 同じ会話の続きとして State を復元できる
- 違う `thread_id` → 別の会話として扱われる

interrupt → resume のフローでは、**同じ `thread_id`** を使うことが必須です。
別の ID を使うと「中断した会話」が見つからず再開できません。

---
## 5. HITL の全体フロー

```
Step 1: agent.invoke({"messages": [...]}, config, version="v2")
   │
   ├─ エージェントが delete_data を呼ぼうとする
   ├─ ミドルウェアが検知 → interrupt 発生!
   ├─ State が checkpointer に保存される
   └─ result.interrupts に中断情報が入って返る

   ここで人間が判断する（approve / edit / reject）

Step 2: agent.invoke(Command(resume={...}), config, version="v2")
   │
   ├─ checkpointer から State を復元
   ├─ 判断に応じて処理:
   │    approve → そのまま実行
   │    edit    → 引数を書き換えて実行
   │    reject  → 実行せず、理由をエージェントに伝える
   └─ エージェントが最終応答を返す
```

**`version="v2"`** を指定すると、戻り値が `GraphOutput` オブジェクトになり、
`.interrupts` 属性で中断情報にアクセスできます。

---
## 6. パターン1: approve（承認）

ツール呼び出しを**そのまま許可**します。
引数の変更なし、エージェントが意図した通りに実行されます。

In [23]:
from langchain.messages import HumanMessage
from langgraph.types import Command

config_approve = {"configurable": {"thread_id": "approve-demo"}}

print("=" * 60)
print("パターン1: approve（承認）")
print("=" * 60)

# --------------------------------------------------
# Step 1: エージェントを実行
#
# agent.invoke() は以下の流れで動く:
#   1. HumanMessage をモデルに渡す
#   2. モデルが delete_data を呼ぶと判断 → AIMessage(tool_calls=[...]) を返す
#   3. ミドルウェアが「delete_data は承認必要」と検知
#   4. interrupt を発生させて実行を一時停止
#   5. State を checkpointer に保存
#   6. result.interrupts に中断情報を入れて返す
#
# もしモデルがツールを呼ばず「本当に削除しますか？」と
# テキストで返した場合、interrupt は発生しない
# --------------------------------------------------
print("\n--- Step 1: エージェント実行 ---")
result = agent.invoke(
    {"messages": [HumanMessage("test_data を削除して")]},
    config=config_approve,
    version="v2",
)

# --------------------------------------------------
# 中断が発生したか確認
#
# result.interrupts:
#   - リストが空でない → ミドルウェアがツール呼び出しを検知して中断した
#   - リストが空       → モデルがツールを呼ばなかった（テキスト応答のみ）
# --------------------------------------------------
if result.interrupts:
    interrupt = result.interrupts[0]
    print(f"  中断発生!")
    print(f"  ツール名: {interrupt.value['action_requests'][0]['name']}")
    print(f"  引数:     {interrupt.value['action_requests'][0]['args']}")

    # --------------------------------------------------
    # Step 2: approve（承認）で再開
    #
    # Command(resume=...) で checkpointer から State を復元し、
    # 中断したところから処理を再開する
    # 同じ thread_id を使うことで「中断した会話の続き」と認識される
    #
    # decisions はリストで渡す
    # → 1つのツール呼び出しに対して1つの判断
    # → 複数ツールが同時に中断された場合は複数の判断を渡す
    # --------------------------------------------------
    print("\n--- Step 2: approve（承認）で再開 ---")
    result = agent.invoke(
        Command(resume={"decisions": [{"type": "approve"}]}),
        config=config_approve,  # 同じ thread_id!
        version="v2",
    )
    print(f"\n最終応答: {result.value["messages"][-1].content}")

else:
    # モデルがツールを呼ばなかった場合
    # → すでに最終応答が返っているのでそのまま表示
    print("  中断なし — モデルがツールを呼ばずにテキストで応答しました")
    print(f"\n最終応答: {result.value["messages"][-1].content}")
    print("\n  ヒント: system_prompt で「必ずツールを使って削除を実行しなさい」と")
    print("  指示すると、ツール呼び出しが発生しやすくなります")

パターン1: approve（承認）

--- Step 1: エージェント実行 ---
  中断発生!
  ツール名: delete_data
  引数:     {'target': 'test_data'}

--- Step 2: approve（承認）で再開 ---

最終応答: 'test_data' を削除しました。何か他にご希望がありましたらお知らせください。


---
## 7. パターン2: reject（拒否）

ツール呼び出しを**拒否**します。
`message` で理由を伝えると、エージェントはその理由を踏まえて応答を返します。

In [22]:
config_reject = {"configurable": {"thread_id": "reject-demo"}}

print("=" * 60)
print("パターン2: reject（拒否）")
print("=" * 60)

# Step 1: エージェント実行 → 中断
print("\n--- Step 1: エージェント実行 → 中断 ---")
result = agent.invoke(
    {"messages": [HumanMessage("important_logs を削除して")]},
    config=config_reject,
    version="v2",
)

if result.interrupts:
    interrupt = result.interrupts[0]
    print(f"  中断発生!")
    print(f"  内容: {interrupt.value}")

# Step 2: reject（拒否）+ 理由メッセージ
print("\n--- Step 2: reject（拒否）で再開 ---")
print("  理由: 'ログは削除しないでください。保持が必要です。'")
result = agent.invoke(
    Command(resume={
        "decisions": [{
            "type": "reject",
            "message": "ログは削除しないでください。保持が必要です。",
        }]
    }),
    config=config_reject,
    version="v2",
)

# reject した場合、エージェントは理由を踏まえた応答を返す
print(f"\n最終応答: {result.value["messages"][-1].content}")

パターン2: reject（拒否）

--- Step 1: エージェント実行 → 中断 ---

--- Step 2: reject（拒否）で再開 ---
  理由: 'ログは削除しないでください。保持が必要です。'

最終応答: 申し訳ありませんが、important_logsは削除が許可されていないため、削除できません。他にお手伝いできることがあれば教えてください。


---
## 8. パターン3: edit（編集）

ツール呼び出しの**引数を書き換えてから実行**します。
例: エージェントが `old_cache` を消そうとしたが、`expired_cache_only` に変更する。

In [21]:
config_edit = {"configurable": {"thread_id": "edit-demo"}}

print("=" * 60)
print("パターン3: edit（編集）")
print("=" * 60)

# Step 1: エージェント実行 → 中断
print("\n--- Step 1: エージェント実行 → 中断 ---")
result = agent.invoke(
    {"messages": [HumanMessage("old_cache を削除して")]},
    config=config_edit,
    version="v2",
)

if result.interrupts:
    interrupt = result.interrupts[0]
    print(f"  中断発生!")
    print(f"  内容: {interrupt.value}")

# Step 2: edit（編集）→ 引数を書き換えて実行
print("\n--- Step 2: edit（編集）で再開 ---")
print("  変更: target='old_cache' → 'expired_cache_only'")
result = agent.invoke(
    Command(resume={
        "decisions": [{
            "type": "edit",
            "edited_action": {
                "name": "delete_data",                   # ツール名
                "args": {"target": "expired_cache_only"}, # 修正した引数
            },
        }]
    }),
    config=config_edit,
    version="v2",
)

print(f"\n最終応答: {result.value["messages"][-1].content}")

パターン3: edit（編集）

--- Step 1: エージェント実行 → 中断 ---
  中断発生!
  内容: {'action_requests': [{'name': 'delete_data', 'args': {'target': 'old_cache'}, 'description': "Tool execution requires approval\n\nTool: delete_data\nArgs: {'target': 'old_cache'}"}], 'review_configs': [{'action_name': 'delete_data', 'allowed_decisions': ['approve', 'edit', 'reject']}]}

--- Step 2: edit（編集）で再開 ---
  変更: target='old_cache' → 'expired_cache_only'

最終応答: expired_cache_only を削除しました。ほかに削除したいデータはありますか？


---
## 9. 安全なツールは中断されない確認

`get_weather` は `interrupt_on` で `False` に設定しているので、
承認なしでそのまま実行されます。

In [20]:
config_safe = {"configurable": {"thread_id": "safe-demo"}}

print("=" * 60)
print("安全なツール: 中断なしで実行")
print("=" * 60)

result = agent.invoke(
    {"messages": [HumanMessage("東京の天気を教えて")]},
    config=config_safe,
    version="v2",
)

# interrupts は空のはず
if result.interrupts:
    print("中断が発生しました（想定外）")
else:
    print("中断なし — 承認不要のツールはそのまま実行されます")

print(f"\n最終応答: {result.value["messages"][-1].content}")

安全なツール: 中断なしで実行
中断なし — 承認不要のツールはそのまま実行されます

最終応答: 東京の天気は晴れで、気温は25度Cです。続けて他の都市の天気やご質問があれば教えてください。


---
## 10. 中断情報の詳細を見る

`result.interrupts` の中身を詳しく確認してみましょう。
実際のアプリでは、この情報を UI に表示して人間に判断を求めます。

In [24]:
import json

config_inspect = {"configurable": {"thread_id": "inspect-demo"}}

result = agent.invoke(
    {"messages": [HumanMessage("temp_files を削除して")]},
    config=config_inspect,
    version="v2",
)

if result.interrupts:
    print("=== interrupt の中身 ===")
    for i, interrupt in enumerate(result.interrupts):
        print(f"\n--- interrupt[{i}] ---")
        print(f"  type : {type(interrupt).__name__}")
        print(f"  value: {interrupt.value}")

    print("\n=== この情報を元に人間が判断する ===")
    print("  → approve: そのまま実行")
    print("  → reject:  実行拒否（理由を伝える）")
    print("  → edit:    引数を修正して実行")

    # 今回は approve で再開
    result = agent.invoke(
        Command(resume={"decisions": [{"type": "approve"}]}),
        config=config_inspect,
        version="v2",
    )
    print(f"\n最終応答: {result.value["messages"][-1].content}")

=== interrupt の中身 ===

--- interrupt[0] ---
  type : Interrupt
  value: {'action_requests': [{'name': 'delete_data', 'args': {'target': 'temp_files'}, 'description': "Tool execution requires approval\n\nTool: delete_data\nArgs: {'target': 'temp_files'}"}], 'review_configs': [{'action_name': 'delete_data', 'allowed_decisions': ['approve', 'edit', 'reject']}]}

=== この情報を元に人間が判断する ===
  → approve: そのまま実行
  → reject:  実行拒否（理由を伝える）
  → edit:    引数を修正して実行

最終応答: temp_files を削除しました。ほかに何かお手伝いできることはありますか？


---
## 11. メッセージ履歴を確認

HITL のフローでメッセージ履歴がどうなっているか確認します。
各ステップでどんなメッセージが追加されたかが分かります。

In [27]:
import uuid

config_trace = {"configurable": {"thread_id": f"trace-{uuid.uuid4().hex[:8]}"}}

# Step 1
result = agent.invoke(
    {"messages": [HumanMessage("debug_logs を削除して")]},
    config=config_trace,
    version="v2",
)

print("=== Step 1 後のメッセージ履歴 ===")
for i, msg in enumerate(result.value["messages"]):
    msg_type = type(msg).__name__
    content = str(msg.content)[:80] if msg.content else "(空)"
    tool_calls = ""
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        names = [tc["name"] for tc in msg.tool_calls]
        tool_calls = f" [ツール呼び出し: {', '.join(names)}]"
    print(f"  [{i}] {msg_type}: {content}{tool_calls}")

# Step 2: approve
result = agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config_trace,
    version="v2",
)

print("\n=== Step 2（approve後）のメッセージ履歴 ===")
for i, msg in enumerate(result.value["messages"]):
    msg_type = type(msg).__name__
    content = str(msg.content)[:80] if msg.content else "(空)"
    tool_calls = ""
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        names = [tc["name"] for tc in msg.tool_calls]
        tool_calls = f" [ツール呼び出し: {', '.join(names)}]"
    print(f"  [{i}] {msg_type}: {content}{tool_calls}")

=== Step 1 後のメッセージ履歴 ===
  [0] HumanMessage: debug_logs を削除して
  [1] AIMessage: (空) [ツール呼び出し: delete_data]

=== Step 2（approve後）のメッセージ履歴 ===
  [0] HumanMessage: debug_logs を削除して
  [1] AIMessage: (空) [ツール呼び出し: delete_data]
  [2] ToolMessage: 'debug_logs' を削除しました
  [3] AIMessage: 'debug_logs' を削除しました。他にご用件はありますか？


---
##12.人間の入力で操作する例

In [28]:
config_interactive = {"configurable": {"thread_id": "interactive-demo"}}

# Step 1: エージェント実行 → 中断
result = agent.invoke(
    {"messages": [HumanMessage("test_data を削除して")]},
    config=config_interactive,
    version="v2",
)

if result.interrupts:
    interrupt = result.interrupts[0]
    print("=" * 50)
    print("承認待ち!")
    print(f"  ツール: {interrupt.value['action_requests'][0]['name']}")
    print(f"  引数:   {interrupt.value['action_requests'][0]['args']}")
    print("=" * 50)

    # 操作者に入力を求める
    choice = input("判断を入力 [approve / reject / edit]: ").strip().lower()

    if choice == "approve":
        decisions = [{"type": "approve"}]

    elif choice == "reject":
        reason = input("拒否理由: ")
        decisions = [{"type": "reject", "message": reason}]

    elif choice == "edit":
        new_target = input("新しい target 値: ")
        decisions = [{
            "type": "edit",
            "edited_action": {
                "name": "delete_data",
                "args": {"target": new_target},
            },
        }]
    else:
        print(f"不明な入力: {choice} → reject として扱います")
        decisions = [{"type": "reject", "message": "無効な入力のため拒否"}]

    # Step 2: 判断を渡して再開
    result = agent.invoke(
        Command(resume={"decisions": decisions}),
        config=config_interactive,
        version="v2",
    )

    print(f"\n最終応答: {result['messages'][-1].content}")

承認待ち!
  ツール: delete_data
  引数:   {'target': 'test_data'}
判断を入力 [approve / reject / edit]: reject
拒否理由: no reason

最終応答: 申し訳ありませんが、"test_data" の削除はできませんでした。ほかにお手伝いできることはありますか？


/tmp/ipykernel_4645/1166472959.py:48: LangGraphDeprecatedSinceV11: Accessing GraphOutput via `result[key]` is deprecated. Use `result.value` to access the output value directly, or `result.interrupts` for interrupts. Deprecated in LangGraph V1.1 to be removed in V3.0.
  print(f"\n最終応答: {result['messages'][-1].content}")


---
## まとめ

### 構成要素の整理

| 要素 | 役割 |
|---|---|
| `HumanInTheLoopMiddleware` | どのツールで中断するかを定義 |
| `checkpointer` | 中断時の State を保存・復元する |
| `thread_id` | どのセッションの State かを識別する |
| `version="v2"` | `.interrupts` 属性が使える戻り値形式 |
| `Command(resume=...)` | 中断からの再開時に判断を伝える |

### 3つの判断

| 判断 | 効果 | resume の書き方 |
|---|---|---|
| `approve` | そのまま実行 | `{"type": "approve"}` |
| `reject` | 実行拒否+理由をエージェントに伝える | `{"type": "reject", "message": "理由"}` |
| `edit` | 引数を修正して実行 | `{"type": "edit", "edited_action": {"name": "...", "args": {...}}}` |

### 注意点

- `checkpointer` がないと HITL は動かない（State を保存できないため）
- Step 1 と Step 2 で**同じ `thread_id`** を使うこと
- `decisions` はリストで渡す（複数ツールが同時に中断された場合に対応）